In [1]:
import numpy as np

import sys
sys.path.insert(1, '../../human_me/')
from human_me import preprocess

from human_me.preprocess import correct_inputs as ci
from human_me.io import load_metabolic_model

full model

In [4]:
# prebuild = '/data2/hratch/human_me/prebuild/'
# ci.correct_model(model = prebuild + 'recon2_2.xml')
# non_machinery, revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
#                             non_machinery = {'HGNC:4556':['m', 'c'], 'HGNC:9251': ['l'], 
#                                             'HGNC:32043': ['e', 'n']})
# print(revised_genes)
# from human_me.expression import build_me_model
# me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False)
# me_model.pickle('/data2/hratch/human_me/full_12_29_20.pickle')

toy model

In [5]:
# other = '/data2/hratch/human_me/other/'
# ci.correct_model(model = other + 'toy_model.xml')
# revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
#                                non_machinery = None)
# print(revised_genes)
# from human_me.expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                             unmodeled_protein_frac = None)
# toy_me_model.pickle(other + 'toy_me_12_29_20.pickle')
# sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

core model

In [2]:
# other_path = '/data2/hratch/human_me/other/'
# mem = False # minimial media
# fn = '/data2/hratch/human_me/other/core' 
# if mem:
#     fn += '_mem'

    
# m_model = load_metabolic_model(fn + '.xml')
# cm_1, cm_2, cm_3 = ci.correct_model(model_file = fn + '.xml')
# # revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
# #                                non_machinery = None)

# # from human_me.expression import build_me_model
# # toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
# #                                             unmodeled_protein_frac = None)
# # toy_me_model.pickle(other + 'toy_me_12_29_20.pickle')
# # sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

# Preprocessing

Throughout preprocessing, we want to make sure that the metabolic model remains feasible for growth. If the metabolic model is not feasible, the ME Model will not be.

In [2]:
import sys
sys.path.insert(1, '/home/hratch/Projects/human_me/')

from human_me import io
fn = '/data2/hratch/human_me/other/central_v2' 
m_model = io.load_metabolic_model(model_file = fn + '.xml') # central carbon metabolism only

In [19]:
from human_me import io

from human_me.utils.parameters import biomass_parameters
from human_me.core import biomass

from human_me.build.build_me_model import build_me

First, check if the metabolic model you plan to input is feasible:

In [4]:
# set metabolic model objective to growth
m_model.objective = 'biomass_reaction'
# check that the model can grow
sln = m_model.slim_optimize()
if sln <= 0 or np.isnan(sln):
    print('The model cannot grow')

Next, check if the biomass objective is formulated correctly. In recon2.2, the lipid and DNA component formation reactions are not formulated correctly. See the Biomass Objective notes for details. In short, we expect stoichiometric coefficients scaled by their molecular weight to sum to 1. Our check function expects biomass reactions to be formatted as they are in recon2.2 (especially the reaction and metabolite IDs).

If you are using the exact Recon2.2 biomass formulation, this will raise warnings:

In [5]:
biomass.check_m_biomass(m_model)

If biomass is not formulated correctly (above cell gave a warning), we strongly recommend using our correcting function. Our correcting function will set the biomass objective to match the default one of the ME Model and remove mass balance issues from Recon2.2. While this is not necessary for building the ME Model, it allows us to check that the corrected metabolic model is still feasible.

Again, this expects biomass reactions' formatting to match Recon2.2

For downstream comparisons between the metabolic model and the ME Model, we recommend using cm_1.

If you corrected, check again that the model remains feasible:

In [6]:
m_model = biomass.correct_m_biomass(m_model)
sln = m_model.slim_optimize()
if sln <= 0 or np.isnan(sln):
    print('The model cannot grow')

Next, apply the other preprocessing corrections to the metabolic model. These are mainly other mistakes found in Recon2.2:

In [7]:
cm_1, cm_2, cm_3 = ci.correct_model(model_file = m_model)

../../human_me/human_me/preprocess/correct_inputs.py:128 UserWarning: PFK contains redundant complexes according to GPR, editing GPR


Check for the recon2.2 HGNC:HGNC error
Remove genes not participating in reactions


../../human_me/human_me/preprocess/correct_inputs.py:207 UserWarning: gpi_hs does not exist in model. Adding to compartment r via sink This allows gpi_hs to be in the model at no cost.
../../human_me/human_me/preprocess/correct_inputs.py:207 UserWarning: uacgam does not exist in model. Adding to compartment g via sink This allows uacgam to be in the model at no cost.
../../human_me/human_me/preprocess/correct_inputs.py:207 UserWarning: udpacgal does not exist in model. Adding to compartment g via sink This allows udpacgal to be in the model at no cost.


In [8]:
for cm in [cm_1, cm_2]:
    sln = cm.slim_optimize()
    if sln <= 0 or np.isnan(sln):
        print('The model cannot grow')
# save the input to me model building
io.write_pickled_object(cm_3, '/home/hratch/Projects/human_me/test_code/mimv2.pickle')

If you want, you can also check that the solution did not change:

In [9]:
np.allclose(m_model.slim_optimize(), cm_1.slim_optimize(), cm_2.slim_optimize())

True

You can also check that the cobrapy glpk solver and the ME Model qMINOS solver arrive at the same solution:

In [16]:
from human_me.me_solver.solve_me import qminosSolver
from cobra.util.solver import linear_reaction_coefficients

objective = {r.id: coef for r, coef in linear_reaction_coefficients(cm_1).items()}
qminos_sln,_,_ = qminosSolver().solve_lp(me_model = m_model, 
                                     mu_val = None, 
                                     objective = objective)

reaction_indeces = [m_model.reactions.index(r_id) for r_id in list(objective)]
np.allclose(m_model.slim_optimize(), qminos_sln[reaction_indeces])

We also need to do some preprocessing on the PSIM:

In [17]:
non_machinery = None
psim_me, non_machinery, revised_genes = ci.correct_psim(me_input_model = cm_3)
# save the input psim
io.write_pickled_object(psim_me, '/home/hratch/Projects/human_me/test_code/psim2.pickle')

# ME Model Building

The ME Model can be built with a single command. The last step, adding gene objects, is very time consuming so we recommend parallelizing (set the n_cores argument).

In [20]:
# the me model can be built with one command
me_model, builder = build_me(me_input_model = cm_3, psim_me = psim_me, n_cores = 30)

# # save the me model
# me_model.pickle('/data2/hratch/human_me/other/test_lp/central_ME.pickle')

Generate ubiquitin reactions for proteasomal degradation
Generate ribosome
Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 556/556 [00:12<00:00, 46.24it/s]


No. iterations for new expression machinery: 1


100%|██████████| 571/571 [00:16<00:00, 34.30it/s]


Express dummy protein
Get metabolic module complex information


...

Deorphan enzymeless reactions
2717 of 3388 protein degradation reactions will be removed because they are not associated with an active enzyme
Couple enzyme degradation to catalysis


100%|██████████| 5988/5988 [00:21<00:00, 281.59it/s]


Add biomass component to reactions
Generate ME-Model
Check reaction mass balances
Make sure all reactions received correct coupled machinery


100%|██████████| 8837/8837 [00:20<00:00, 428.84it/s] 


Add gene objects
Time to build: 5.49 minutes


Note, if you are going to reload the model from it's saved pickle format, make sure to use the human_me.io function to do so:

In [ ]:
io.read_pickled_me_model('/data2/hratch/human_me/other/test_lp/central_ME.pickle')

## ME Model Biomass Mass Fraction Sanity Check

Once we have generated the ME Model, we can check that the biomass formation reactions were implemented correctly. To do so, we must check that the expected mass fractions match the input mass fractions.The expected mass fraction is calculated as the sum of the stoichiometric coefficient * molecular weight of all the metabolites used in forming that biomass component. If the expected mass fraction does not match the input mass fraction, something is wrong either with the input mass fractions or the input coefficients for the biomass formation reactions 

In [16]:
expected_mass_fraction = biomass.check_me_biomass(me_model)
expected_mass_fraction

{'DNA': 0.014054153934054002,
 'carbohydrate': 0.07103201555619006,
 'lipid': 0.09699999999999999}

In our case, we used the default mass fraction values as input:

In [17]:
mass_fraction = biomass_parameters.mass_fraction
mass_fraction

{'DNA': 0.014, 'carbohydrate': 0.071, 'lipid': 0.097, 'other': 0.054}

Finally, we can check if the expected mass fraction matched the input one (values should be ~0):

In [18]:
difference = dict()
for biomass_type, mf in expected_mass_fraction.items():
    difference[biomass_type] = abs(mf - mass_fraction[biomass_type])
difference

{'DNA': 5.415393405400204e-05,
 'carbohydrate': 3.201555619006258e-05,
 'lipid': 1.3877787807814457e-17}

Next, we can check that the ME Model can grow. To do so, we can set a very small value for growth and see if the ME Model is feasible. If it returns a status of zero, it is feasible

In [21]:
sln, stat, _ = me_model.solve_lp(mu_val = 1e-9)

../../human_me/human_me/core/model.py:422 UserWarning: Solver is not initialized with ME_Model.intialize_solver, intializing with default parameters


Getting MINOS parameters...
Done in 88.9231 seconds with status 0


## Optimizing for an Objective

# Trash

In [29]:
sln_2, stat, _ = me_model.solve_lp(mu_val = 1e-9)

Getting MINOS parameters...
Done in 88.3758 seconds with status 0


In [23]:
# io.write_pickled_object(me_model, '/data2/hratch/human_me/other/test_lp/central_ME.pickle')
# io.write_pickled_object(sln, '/data2/hratch/human_me/other/test_lp/feasible_sln.pickle')

# import pandas as pd
# fluxes = pd.DataFrame(index = [r.id for r in me_model.reactions], 
#                       data = {'fluxes_1': sln[:len(me_model.reactions)], 
#                                'fluxes_2': sln_2[:len(me_model.reactions)]})
# fluxes.to_csv('/data2/hratch/human_me/other/test_lp/feasible_sln.csv')

In [ ]:
# 0-1: repeats of original, 2: change name to orphan, 3: actually make flux through orphan, 
# 4: feed orphan/known into modeled protein
# 5: unmodeled implemented, Q = 0

In [71]:
# fluxes = pd.read_csv('/data2/hratch/human_me/other/test_lp/feasible_sln.csv', index_col = 0)
fluxes_ = pd.read_csv('/data2/hratch/human_me/other/test_lp/sln.csv', index_col = 0)

fluxes = pd.concat([fluxes, fluxes_], axis = 1)
fluxes.columns = ['fluxes_' + str(i) for i in range(fluxes.shape[1])]
# fluxes.to_csv('/data2/hratch/human_me/other/test_lp/feasible_sln.csv')

In [72]:
biomass_reactions = [r_id for r_id in fluxes.index if 'biomass' in r_id or 'DUMMY' in r_id]
fluxes.loc[biomass_reactions,:]

,fluxes_0,fluxes_1,fluxes_2,fluxes_3,fluxes_4,fluxes_5,fluxes_6
HGNC:DUMMY_lariats_DEGRADATIONn,6.657039e-17,6.657039e-17,NaN,NaN,NaN,NaN,NaN
HGNC:DUMMY_DECAPPING_mRNA_DEGRADATIONc,3.328519e-17,3.328519e-17,NaN,NaN,NaN,NaN,NaN
HGNC:DUMMY_TRANSCRIPTION,6.657039e-17,6.657039e-17,NaN,NaN,NaN,NaN,NaN
HGNC:DUMMY_TRANSLATION_ELONGATIONc,2.092821e-12,2.092821e-12,NaN,NaN,NaN,NaN,NaN
HGNC:DUMMY_CYTOSOLIC_PROTEIN_FOLDING,2.092821e-12,2.092821e-12,NaN,NaN,NaN,NaN,NaN
HGNC:DUMMY_folded_protein_c_POLYUBIQUITINATIONc,1.046410e-12,1.046410e-12,NaN,NaN,NaN,NaN,NaN
HGNC:DUMMY_folded_protein_c_DEUBIQUITINATIONc,0.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN
HGNC:DUMMY_folded_protein_c_PROTEASOMAL_DEGRADATIONc,1.046410e-12,1.046410e-12,NaN,NaN,NaN,NaN,NaN
biomass_dilution,1.000000e-09,1.000000e-09,1.000000e-09,1.000000e-09,1.000000e-09,1.000000e-09,1.000000e-09
DNA_biomass_to_biomass,1.400000e-11,1.400000e-11,1.400000e-11,1.400000e-11,1.400000e-11,1.400000e-11,1.400000e-11


In [34]:
[m for m in me_model.metabolites if hasattr(m, 'dummy') and m.dummy]

[<Protein HGNC:DUMMY_unfolded_protein_c at 0x7ff7a7a79518>,
 <Protein HGNC:DUMMY_folded_protein_c at 0x7ff7a7a79630>,
 <Protein HGNC:DUMMY_folded_protein_polyub_protein_c at 0x7ff7a7a796a0>]

In [39]:
me_model.metabolites.get_by_id('HGNC:DUMMY_unfolded_protein_c').reactions

frozenset({<ProteinExpressionReaction HGNC:DUMMY_CYTOSOLIC_PROTEIN_FOLDING at 0x7ff77e5b4ba8>,
           <ProteinExpressionReaction HGNC:DUMMY_CYTOSOLIC_PROTEIN_FOLDING at 0x7ff7c20024a8>,
           <ProteinExpressionReaction HGNC:DUMMY_TRANSLATION_ELONGATIONc at 0x7ff7c20632b0>,
           <ProteinExpressionReaction HGNC:DUMMY_TRANSLATION_ELONGATIONc at 0x7ff77e5b48d0>})

In [42]:
me_model.reactions.get_by_id('HGNC:DUMMY_TRANSLATION_ELONGATIONc').reaction

'(mu + 0.0438248147682957)/(162090.334890627*mu + 2755.50353145864) HGNC:DUMMY_mrna_c + 0.0438248147682957/(162090.334890627*mu + 2755.50353145864) HGNC:DUMMY_mrna_degradation_proxy_c + 2.90404420365785e6*mu  2.9581855341595e8 TRANSLATION_ELONGATIONc_complex_c + 68 charged_generic_A_trna_c + 17 charged_generic_C_trna_c + 43 charged_generic_D_trna_c + 54 charged_generic_E_trna_c + 42 charged_generic_F_trna_c + 65 charged_generic_G_trna_c + 22 charged_generic_H_trna_c + 48 charged_generic_I_trna_c + 49 charged_generic_K_trna_c + 96 charged_generic_L_trna_c + 23 charged_generic_M_trna_c + 32 charged_generic_N_trna_c + 47 charged_generic_P_trna_c + 34 charged_generic_Q_trna_c + 47 charged_generic_R_trna_c + 60 charged_generic_S_trna_c + 46 charged_generic_T_trna_c + 63 charged_generic_V_trna_c + 13 charged_generic_W_trna_c + 29 charged_generic_Y_trna_c + 898 gtp_c + 899 h2o_c --> HGNC:DUMMY_unfolded_protein_c + 100.210245180000 biomass_protein + 898 gdp_c + 898 generic_trna_c + 1796 h_c + 

In [44]:
fluxes.loc['HGNC:DUMMY_TRANSLATION_ELONGATIONc', :]

fluxes_1    2.092821e-12
fluxes_2    2.092821e-12
Name: HGNC:DUMMY_TRANSLATION_ELONGATIONc, dtype: float64